# Exercise wearable audit for AI-READI

This notebook audits the AI-READI processed and model-ready files to identify which
wearable signals are available and reliable for exercise-like episode detection.

**Goal:** Determine which of the following analysis paths is supported by available data:

1. **Exercise-like episode detection** -- if true activity/steps plus heart rate are available.
2. **Physiological-arousal response modeling** -- if heart rate, sleep/awake, SpO2, stress,
   or respiratory rate are available but true activity is missing.
3. **No exercise analysis** -- if wearable data are too sparse or unreliable.

**Signals inspected:** heart rate, respiratory rate, steps/activity, calories,
sleep/awake state, stress, and SpO2.

**Important constraints for this notebook:**
- Exercise episodes are NOT detected here.
- Future glucose is NOT analyzed here.
- Glucose is NOT used to define exercise.
- Only column inspection, missingness, distributions, usability, and signal quality are assessed.
- Each signal is evaluated for whether it is causal (can be observed without knowing the outcome)
  and time-varying within participants.


## Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow.parquet as pq
import pyarrow as pa
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.backends.backend_pdf as mpdf
import json
import warnings
import os
from collections import defaultdict

warnings.filterwarnings("ignore")

try:
    import duckdb
    HAS_DUCKDB = True
    print(f"duckdb available: {duckdb.__version__}")
except ImportError:
    HAS_DUCKDB = False
    print("duckdb not available -- using pandas/pyarrow fallback")


duckdb not available -- using pandas/pyarrow fallback


## 1. Configuration

Set `DATA_PATH` and `STATIC_PATH` to override auto-discovery.
Leave as `None` to let the notebook search for candidate files automatically.
`SAMPLE_N` controls the maximum number of rows loaded for per-column metrics;
increase it if you have enough RAM.


In [2]:
# ---------------------------------------------------------------
# Configuration -- change only this cell to point at other files
# ---------------------------------------------------------------
DATA_PATH   = None   # Path to main longitudinal/dynamic parquet
STATIC_PATH = None   # Path to participant static features parquet
OUTPUT_DIR  = "outputs/exercise_wearable_audit"
SAMPLE_N    = 500_000  # max rows to load for per-column metrics

OUTPUT_DIR = Path(OUTPUT_DIR)
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR.resolve()}")


Output directory: /home/myriamcharfeddine/CGM/SSM-CGM/notebooks/outputs/exercise_wearable_audit


## 2. Modality keyword lists and helper functions

The lists below are used throughout the notebook to classify columns into modalities
without hard-coding column names.  Adjust the lists if the dataset uses different naming.


In [3]:
MODALITY_KEYWORDS = {
    "glucose":     ["glucose", "cgm", "blood_glucose", "dexcom"],
    "heart_rate":  ["heart_rate", "heartrate", "hr", "pulse"],
    "resp_rate":   ["respiratory", "respiration", "resp_rate", "rr", "breath"],
    "steps":       ["steps", "step", "activity", "active", "sedentary",
                    "distance", "movement"],
    "calories":    ["calorie", "calories", "energy", "kcal"],
    "sleep":       ["sleep", "awake", "rem", "deep", "light", "asleep", "bed"],
    "stress":      ["stress", "eda", "conductance"],
    "spo2":        ["spo2", "oxygen", "ox", "saturation", "pulse_ox"],
    "time":        ["time", "datetime", "timestamp", "date", "minute",
                    "hour", "anchor", "ds"],
    "participant": ["participant", "subject", "person", "segment", "stream"],
    "clinical":    ["hba1c", "bmi", "insulin", "c_peptide", "cpeptide",
                    "site", "study_group", "med", "drug", "age", "sex",
                    "demo", "race", "ethnicity", "marital", "pregnant",
                    "cholesterol", "triglyceride", "waist", "bp", "diastolic",
                    "systolic", "baseline"],
}

# Physiological plausible ranges (broad, for flagging obvious artifacts)
PHYS_RANGES = {
    "heart_rate": (30, 220),
    "resp_rate":  (5, 50),
    "spo2":       (50, 100),
    "glucose":    (40, 400),
}

# Missingness threshold above which a signal is considered too sparse
MISSING_THRESHOLD = 70.0

# Minimum number of participants with at least 10 usable rows
MIN_PARTICIPANTS_USABLE = 5


def classify_column(col_name: str) -> list:
    """Return list of modality labels that match a column name.

    Short keywords (2 chars or fewer) require an exact underscore-delimited token match
    to avoid false positives (e.g. 'rr' inside 'currently', 'ds' inside 'records').
    Longer keywords use substring matching against the full lowercased column name.
    """
    import re as _re
    col_lower = col_name.lower()
    col_tokens = set(_re.split(r'[_\-\.\s]+', col_lower))
    matches = []
    for modality, keywords in MODALITY_KEYWORDS.items():
        for kw in keywords:
            kw_lower = kw.lower()
            if len(kw_lower) <= 2:
                hit = kw_lower in col_tokens
            else:
                hit = kw_lower in col_lower
            if hit:
                matches.append(modality)
                break
    return matches if matches else ["unknown"]


## 3. File discovery

The notebook searches recursively from the current working directory (and up to 4 parent levels)
for parquet files that may be the AI-READI processed dataset.
It ranks candidates by file size and by name pattern.


In [4]:
def find_parquet_files(max_parents=4):
    """Recursively search for parquet files near the working directory."""
    cwd = Path.cwd()
    search_roots = [cwd]
    parent = cwd
    for _ in range(max_parents):
        parent = parent.parent
        if parent == parent.parent:
            break
        search_roots.append(parent)

    SKIP_DIRS = {".git", "__pycache__", "node_modules", ".venv", "venv",
                 "env", ".ipynb_checkpoints"}
    found = {}
    for root in search_roots:
        try:
            for p in root.rglob("*.parquet"):
                if any(skip in p.parts for skip in SKIP_DIRS):
                    continue
                rp = str(p.resolve())
                if rp in found:
                    continue
                try:
                    found[rp] = {"path": p.resolve(), "size_mb": p.stat().st_size / 1e6}
                except Exception:
                    pass
        except PermissionError:
            pass

    return sorted(found.values(), key=lambda x: x["size_mb"], reverse=True)


candidate_files = find_parquet_files()
print(f"Found {len(candidate_files)} parquet file(s) total.  Showing largest 25:\n")
for cf in candidate_files[:25]:
    print(f"  {cf['size_mb']:8.1f} MB  {cf['path']}")
if len(candidate_files) > 25:
    print(f"  ... and {len(candidate_files) - 25} more")


Found 114 parquet file(s) total.  Showing largest 25:

     770.9 MB  /home/myriamcharfeddine/CGM/SSM-CGM/outputs/study2_forecast_cache_5min/study2_forecast_cache.parquet
     332.7 MB  /home/myriamcharfeddine/CGM/SSM-CGM/outputs/no_log_scenarios/meal_transfer/online_causal/online_meal_states.parquet
     199.3 MB  /home/myriamcharfeddine/CGM/SSM-CGM/outputs/aireadi_stream_mamba_stateful_5epoch_eval_validation/predictions/predictions.parquet
     199.3 MB  /home/myriamcharfeddine/CGM/SSM-CGM/outputs/aireadi_stream_full_1epoch_eval_validation/predictions/predictions.parquet
     199.1 MB  /home/myriamcharfeddine/CGM/SSM-CGM/outputs/aireadi_stream_mamba_stateful_10epoch_eval_validation/predictions/predictions.parquet
     183.4 MB  /home/myriamcharfeddine/CGM/SSM-CGM/outputs/aireadi_stream_mamba_stateful_5epoch_eval_test/predictions/predictions.parquet
     183.4 MB  /home/myriamcharfeddine/CGM/SSM-CGM/outputs/aireadi_stream_full_1epoch_eval_test/predictions/predictions.parquet
     183.

In [5]:
DYNAMIC_KEYWORDS = ["multimodal", "final", "model_ready", "processed",
                    "longitudinal", "enriched"]
STATIC_KEYWORDS  = ["static", "participant_static", "static_features",
                    "demographics", "participant_measurements"]


def auto_select_paths(candidates, data_path=None, static_path=None):
    """Pick the most likely dynamic and static files from the candidate list."""
    dynamic_cands = []
    static_cands  = []
    for cf in candidates:
        nm = cf["path"].name.lower()
        if any(kw in nm for kw in STATIC_KEYWORDS):
            static_cands.append(cf)
        elif any(kw in nm for kw in DYNAMIC_KEYWORDS):
            dynamic_cands.append(cf)

    if not dynamic_cands and candidates:
        dynamic_cands = [candidates[0]]  # fallback: largest file

    chosen_data   = Path(data_path)   if data_path   else (dynamic_cands[0]["path"]  if dynamic_cands  else None)
    chosen_static = Path(static_path) if static_path else (static_cands[0]["path"]   if static_cands   else None)
    return chosen_data, chosen_static


DATA_PATH, STATIC_PATH = auto_select_paths(candidate_files, DATA_PATH, STATIC_PATH)

print(f"Selected DATA_PATH   : {DATA_PATH}")
print(f"Selected STATIC_PATH : {STATIC_PATH}")
if DATA_PATH is None:
    print("\nWARNING: No dynamic data file found.  Set DATA_PATH manually in the config cell.")


Selected DATA_PATH   : /home/myriamcharfeddine/CGM/Data/enriched_multimodal/final_multimodal_dataset_20260515_184339.parquet
Selected STATIC_PATH : /home/myriamcharfeddine/CGM/Data/enriched_multimodal/participant_measurements_selected_long.parquet


## 4. Schema inspection

Load only the Parquet schema (no data rows) to enumerate columns, infer dtypes,
and classify each column into modalities before touching any data.
This avoids reading a large file into memory unnecessarily.


In [6]:
def load_schema(path, label="file"):
    """Return (schema, col_names, col_types_dict, n_rows) for a parquet file."""
    if path is None or not Path(path).exists():
        print(f"  {label}: path not found or not set.")
        return None, [], {}, 0
    try:
        pf   = pq.ParquetFile(path)
        meta = pf.metadata
        sch  = pf.schema_arrow
        names = sch.names
        types = {sch.field(i).name: str(sch.field(i).type) for i in range(len(sch))}
        print(f"  {label}: {meta.num_rows:,} rows  x  {meta.num_columns} columns")
        return sch, names, types, meta.num_rows
    except Exception as exc:
        print(f"  {label}: could not read schema -- {exc}")
        return None, [], {}, 0


print("=" * 60)
print("DYNAMIC DATA FILE")
print("=" * 60)
dyn_schema, dyn_cols, dyn_types, dyn_nrows = load_schema(DATA_PATH, "dynamic")

print()
print("=" * 60)
print("STATIC FEATURES FILE")
print("=" * 60)
sta_schema, sta_cols, sta_types, sta_nrows = load_schema(STATIC_PATH, "static")


DYNAMIC DATA FILE
  dynamic: 4,532,158 rows  x  140 columns

STATIC FEATURES FILE
  static: 33,799 rows  x  16 columns


In [7]:
def classify_all_columns(col_names, col_types):
    info = {}
    for col in col_names:
        modalities = classify_column(col)
        info[col] = {"dtype": col_types.get(col, "unknown"), "modalities": modalities}
    return info


dyn_col_info = classify_all_columns(dyn_cols, dyn_types) if dyn_cols else {}
sta_col_info = classify_all_columns(sta_cols, sta_types) if sta_cols else {}

modality_groups: dict = defaultdict(list)
for col, info in dyn_col_info.items():
    for m in info["modalities"]:
        modality_groups[m].append(col)

print("Dynamic file -- columns grouped by modality:\n")
for mod in sorted(modality_groups):
    cols_m = modality_groups[mod]
    print(f"  [{mod}]  ({len(cols_m)} column(s))")
    for c in cols_m:
        print(f"      {c}  ({dyn_types.get(c, '?')})")

# Convenience lists used in later cells
id_cols    = modality_groups.get("participant", [])
time_cols  = modality_groups.get("time", [])
gluc_cols  = modality_groups.get("glucose", [])
hr_cols    = modality_groups.get("heart_rate", [])
rr_cols    = modality_groups.get("resp_rate", [])
step_cols  = modality_groups.get("steps", [])
cal_cols   = modality_groups.get("calories", [])
sleep_cols = modality_groups.get("sleep", [])
stress_cols= modality_groups.get("stress", [])
spo2_cols  = modality_groups.get("spo2", [])
clin_cols  = modality_groups.get("clinical", [])

print()
print("Identified column groups:")
print(f"  Participant ID : {id_cols}")
print(f"  Time           : {time_cols}")
print(f"  Glucose        : {gluc_cols}")
print(f"  Heart rate     : {hr_cols}")
print(f"  Respiratory    : {rr_cols}")
print(f"  Steps/activity : {step_cols}")
print(f"  Calories       : {cal_cols}")
print(f"  Sleep/awake    : {sleep_cols}")
print(f"  Stress         : {stress_cols}")
print(f"  SpO2           : {spo2_cols}")
print(f"  Clinical/static: {clin_cols[:10]}{'...' if len(clin_cols) > 10 else ''}")


Dynamic file -- columns grouped by modality:

  [calories]  (2 column(s))
      calories_total  (double)
      calories_per_min  (double)
  [clinical]  (96 column(s))
      sleep_stage_awake  (double)
      sleep_stage_light  (double)
      sleep_stage_deep  (double)
      sleep_stage_rem  (double)
      sleep_stage_unknown  (double)
      activity_stage_walking  (double)
      activity_stage_sedentary  (double)
      activity_stage_generic  (double)
      activity_stage_running  (double)
      participants_clinical_site  (string)
      participants_study_group  (string)
      participants_age  (int64)
      bmi_baseline  (double)
      bmi_baseline_date  (timestamp[ns])
      bmi_n_records  (double)
      bmi_value_range  (double)
      bmi_days_to_cgm_start  (double)
      c_peptide_ngml_baseline  (double)
      c_peptide_ngml_baseline_date  (timestamp[ns])
      c_peptide_ngml_n_records  (double)
      c_peptide_ngml_value_range  (double)
      c_peptide_ngml_days_to_cgm_start  (dou

## 5. Efficient sample loading

For per-column statistics we load at most `SAMPLE_N` rows.
If `duckdb` is available, we use `USING SAMPLE` for a random sample;
otherwise pyarrow reads the first N rows in batches.
All metrics computed below are **sample-based** when the file exceeds `SAMPLE_N` rows.


In [8]:
def load_sample(path, n=SAMPLE_N, cols=None):
    """Load up to n rows from a parquet file.  Returns a pandas DataFrame."""
    if path is None or not Path(path).exists():
        return pd.DataFrame()
    path_str = str(path)

    if HAS_DUCKDB:
        try:
            con = duckdb.connect()
            col_sel = ", ".join(f'"{c}"' for c in cols) if cols else "*"
            q = f"SELECT {col_sel} FROM read_parquet('{path_str}') USING SAMPLE {n} ROWS"
            df = con.execute(q).df()
            con.close()
            label = "random sample via duckdb"
            print(f"Loaded {len(df):,} rows ({label}) from {Path(path).name}")
            return df
        except Exception as exc:
            print(f"duckdb sampling failed ({exc}) -- falling back to pyarrow")

    # pyarrow fallback: read first n rows in batches
    try:
        pf = pq.ParquetFile(path)
        batches = []
        rows_read = 0
        for batch in pf.iter_batches(batch_size=100_000, columns=cols):
            batches.append(batch)
            rows_read += len(batch)
            if rows_read >= n:
                break
        tbl = pa.concat_tables([pa.Table.from_batches([b]) for b in batches])
        df  = tbl.slice(0, n).to_pandas()
        print(f"Loaded {len(df):,} rows (first-N via pyarrow) from {Path(path).name}")
        return df
    except Exception as exc:
        print(f"pyarrow loading failed: {exc}")
        return pd.DataFrame()


# Build the list of columns to load (avoid pulling all 140 columns into RAM)
AUDIT_COLS = list(dict.fromkeys(
    id_cols + time_cols + gluc_cols + hr_cols + rr_cols +
    step_cols + cal_cols + sleep_cols + stress_cols + spo2_cols
))
AUDIT_COLS = [c for c in AUDIT_COLS if c in dyn_cols]

print(f"Loading {len(AUDIT_COLS)} wearable / time / id columns ...")
df = load_sample(DATA_PATH, n=SAMPLE_N, cols=AUDIT_COLS if AUDIT_COLS else None)
IS_SAMPLE = len(df) < dyn_nrows
print(f"Sample shape : {df.shape}   (sample-based metrics: {IS_SAMPLE})")
df.head(3)


Loading 77 wearable / time / id columns ...
Loaded 500,000 rows (first-N via pyarrow) from final_multimodal_dataset_20260515_184339.parquet
Sample shape : (500000, 77)   (sample-based metrics: True)


,participant_id,participants_clinical_site,participants_study_group,participants_age,participants_study_visit_date,timezone,timestamp_local,bmi_baseline_date,c_peptide_ngml_baseline_date,clinical_diastolic_bp_mmhg_baseline_date,...,stress_level_min,stress_level_max,stress_level_count,stress_level_device_availability,oxygen_saturation_mean,oxygen_saturation_std,oxygen_saturation_min,oxygen_saturation_max,oxygen_saturation_count,oxygen_saturation_device_availability
0,1023,UW,insulin_dependent,67,2023-08-30,America/Los_Angeles,2023-08-31 00:10:00-07:00,2023-08-30,2023-08-30,2023-08-30,...,10.0,18.0,5.0,1.0,93.0,0.000000,93.0,93.0,4.0,1.0
1,1023,UW,insulin_dependent,67,2023-08-30,America/Los_Angeles,2023-08-31 00:15:00-07:00,2023-08-30,2023-08-30,2023-08-30,...,18.0,23.0,4.0,0.8,93.0,0.000000,93.0,93.0,2.0,1.0
2,1023,UW,insulin_dependent,67,2023-08-30,America/Los_Angeles,2023-08-31 00:20:00-07:00,2023-08-30,2023-08-30,2023-08-30,...,22.0,24.0,5.0,1.0,83.8,4.024922,82.0,91.0,5.0,1.0


## 6. Per-column wearable audit

For every candidate column we compute:

- Missingness percentage
- Zero fraction
- Unique value count
- Basic statistics (min, max, mean, std, median, IQR)
- Number of participants with at least 10 usable values
- Whether the signal is time-varying within participants (vs. constant)
- Whether it is monotonically increasing (possible cumulative counter)
- Whether it increments by exactly 1 each row (possible row-index counter)
- Whether values are mostly integers
- Whether values fall within a plausible physiological range


In [9]:
participant_col = id_cols[0] if id_cols else None
time_col        = time_cols[0] if time_cols else None


def is_monotonic_per_group(series, group_ids):
    """Fraction of participants whose column is monotonically non-decreasing."""
    if group_ids is None:
        return np.nan
    counts = 0
    total  = 0
    for _, grp in pd.concat([group_ids, series], axis=1).groupby(group_ids.name):
        vals = grp.iloc[:, 1].dropna().values
        if len(vals) < 3:
            continue
        total += 1
        if np.all(np.diff(vals) >= 0):
            counts += 1
    return counts / total if total > 0 else np.nan


def is_sequential_counter(series, group_ids):
    """Fraction of participants whose column increments by exactly 1 each step."""
    if group_ids is None:
        return np.nan
    counts = 0
    total  = 0
    for _, grp in pd.concat([group_ids, series], axis=1).groupby(group_ids.name):
        vals = grp.iloc[:, 1].dropna().values
        if len(vals) < 3:
            continue
        total += 1
        diffs = np.diff(vals)
        if np.all(diffs == 1):
            counts += 1
    return counts / total if total > 0 else np.nan


def audit_column(df, col):
    """Return a dict of audit metrics for one column."""
    if col not in df.columns:
        return None

    s = df[col]
    n_total  = len(s)
    n_null   = int(s.isna().sum())
    n_nonnull= n_total - n_null
    miss_pct = 100.0 * n_null / n_total if n_total > 0 else 100.0
    s_valid  = s.dropna()
    is_num   = pd.api.types.is_numeric_dtype(s)

    n_zero    = int((s_valid == 0).sum()) if (is_num and len(s_valid) > 0) else 0
    zero_frac = n_zero / n_nonnull if n_nonnull > 0 else np.nan
    n_unique  = int(s_valid.nunique())

    if is_num and n_nonnull > 0:
        vals    = s_valid.values.astype(float)
        val_min = float(np.nanmin(vals))
        val_max = float(np.nanmax(vals))
        val_mean= float(np.nanmean(vals))
        val_std = float(np.nanstd(vals))
        val_med = float(np.nanmedian(vals))
        val_q25 = float(np.nanpercentile(vals, 25))
        val_q75 = float(np.nanpercentile(vals, 75))
        val_iqr = val_q75 - val_q25
        mostly_int = float(np.mean(vals == np.round(vals))) > 0.90
    else:
        val_min = val_max = val_mean = val_std = val_med = val_q25 = val_q75 = val_iqr = np.nan
        mostly_int = False

    # Participant-level metrics
    n_part_usable   = np.nan
    time_varying    = np.nan
    const_per_part  = np.nan
    possible_cumul  = np.nan
    possible_ctr    = np.nan

    if participant_col and participant_col in df.columns and is_num:
        sub = df[[participant_col, col]].dropna(subset=[col])
        if len(sub) > 0:
            pid_col   = sub[participant_col]
            val_col   = sub[col]
            grp_std   = sub.groupby(participant_col)[col].std()
            n_var     = int((grp_std > 0).sum())
            n_cst     = int((grp_std == 0).sum())
            n_enough  = int((sub.groupby(participant_col)[col].count() >= 10).sum())
            n_part_usable  = n_enough
            time_varying   = n_var > 0
            const_per_part = (n_cst == grp_std.shape[0])

            mono_frac = is_monotonic_per_group(val_col, pid_col)
            seq_frac  = is_sequential_counter(val_col, pid_col)
            possible_cumul = bool(mono_frac is not np.nan and mono_frac > 0.5)
            possible_ctr   = bool(seq_frac  is not np.nan and seq_frac  > 0.3)

    return {
        "dtype":                           str(s.dtype),
        "n_nonnull":                       n_nonnull,
        "missing_pct":                     round(miss_pct, 2),
        "zero_frac":                       round(float(zero_frac), 4) if not np.isnan(zero_frac) else np.nan,
        "n_unique":                        n_unique,
        "min":                             val_min,
        "max":                             val_max,
        "mean":                            round(val_mean, 4) if not np.isnan(val_mean) else np.nan,
        "std":                             round(val_std,  4) if not np.isnan(val_std)  else np.nan,
        "median":                          round(val_med,  4) if not np.isnan(val_med)  else np.nan,
        "iqr":                             round(val_iqr,  4) if not np.isnan(val_iqr)  else np.nan,
        "n_participants_usable":           n_part_usable,
        "time_varying_within_participant": time_varying,
        "constant_per_participant":        const_per_part,
        "possible_cumulative_counter":     possible_cumul,
        "possible_sequential_counter":     possible_ctr,
        "mostly_integer":                  mostly_int,
        "is_numeric":                      is_num,
    }


print("Running per-column audit ...")
audit_results = {}
ALL_CANDIDATE_COLS = list(dict.fromkeys(
    gluc_cols + hr_cols + rr_cols + step_cols + cal_cols +
    sleep_cols + stress_cols + spo2_cols + id_cols + time_cols
))
for col in ALL_CANDIDATE_COLS:
    if col in df.columns:
        audit_results[col] = audit_column(df, col)
    else:
        print(f"  WARNING: '{col}' not in sample -- skipping")

print(f"Audited {len(audit_results)} columns.")


Running per-column audit ...
Audited 77 columns.


## 7. Physiological range checks

Each numeric wearable column is compared against the plausible ranges below.
Columns whose observed min/max fall outside these ranges are flagged as potential artifacts.

| Modality      | Plausible range                              |
|---------------|----------------------------------------------|
| Heart rate    | 30 - 220 bpm                                 |
| Respiratory   | 5 - 50 breaths/min                           |
| SpO2          | 50 - 100 % (or 0 - 1 if fraction scale)     |
| Glucose       | 40 - 400 mg/dL (response analysis only)      |

Steps and calories have no universal physiological range;
their columns are instead assessed for counter patterns and zero inflation.


In [10]:
def check_phys_range(metrics, modality):
    """
    Return True if observed values are plausible, False if out of range,
    None if check is not applicable, np.nan if no data.
    """
    if metrics is None or not metrics.get("is_numeric", False):
        return np.nan
    val_min = metrics.get("min", np.nan)
    val_max = metrics.get("max", np.nan)
    if np.isnan(val_min) or np.isnan(val_max):
        return np.nan

    if modality == "heart_rate":
        lo, hi = PHYS_RANGES["heart_rate"]
        return bool(val_min >= lo * 0.5 and val_max <= hi * 1.1)

    elif modality == "resp_rate":
        lo, hi = PHYS_RANGES["resp_rate"]
        return bool(val_max <= hi * 1.5 and val_max > lo * 0.5)

    elif modality == "spo2":
        if val_max <= 1.01:
            # Fraction scale (0-1) instead of percentage -- flag as scale issue
            return None
        return bool(val_min >= PHYS_RANGES["spo2"][0] * 0.5 and val_max <= 100.5)

    elif modality == "glucose":
        lo, hi = PHYS_RANGES["glucose"]
        return bool(val_max <= hi * 1.5 and val_max > lo * 0.5)

    else:
        return np.nan


# Attach physiological range pass/fail to each audited column
phys_range_pass = {}
for col, metrics in audit_results.items():
    modalities = classify_column(col)
    primary_mod = modalities[0] if modalities else "unknown"
    phys_range_pass[col] = check_phys_range(metrics, primary_mod)

print("Physiological range check summary:")
print(f"  {'Column':<45}  {'Modality':<12}  {'Pass?'}")
print(f"  {'-'*45}  {'-'*12}  {'-'*6}")
for col, passed in phys_range_pass.items():
    mod = (classify_column(col)[0] if classify_column(col) else "?")
    if mod in ("heart_rate", "resp_rate", "spo2", "glucose"):
        icon = "OK" if passed is True else ("SCALE?" if passed is None else "FAIL" if passed is False else "n/a")
        m = audit_results.get(col)
        rng = f"[{m['min']:.1f}, {m['max']:.1f}]" if (m and not np.isnan(m.get('min', np.nan))) else "n/a"
        print(f"  {col:<45}  {mod:<12}  {icon}  range={rng}")


Physiological range check summary:
  Column                                         Modality      Pass?
  ---------------------------------------------  ------------  ------
  cgm_glucose_mean                               glucose       OK  range=[40.0, 400.0]
  cgm_count                                      glucose       FAIL  range=[0.0, 2.0]
  bmi_days_to_cgm_start                          glucose       n/a  range=n/a
  c_peptide_ngml_days_to_cgm_start               glucose       n/a  range=n/a
  clinical_diastolic_bp_mmhg_days_to_cgm_start   glucose       n/a  range=n/a
  clinical_resting_hr_bpm_days_to_cgm_start      glucose       n/a  range=n/a
  clinical_systolic_bp_mmhg_days_to_cgm_start    glucose       n/a  range=n/a
  hba1c_percent_days_to_cgm_start                glucose       n/a  range=n/a
  hdl_cholesterol_mgdl_days_to_cgm_start         glucose       n/a  range=n/a
  ldl_cholesterol_mgdl_days_to_cgm_start         glucose       n/a  range=n/a
  serum_glucose_mgdl_baseline

## 8. Column decision table

Each candidate column is assigned:

- `usable_for_exercise_detection` -- can serve as a primary exercise signal
- `usable_for_response_analysis`  -- can serve as a response or support signal
- `recommended_role`              -- one of the roles listed below

**Role taxonomy:**

| Role                        | Meaning                                                            |
|-----------------------------|--------------------------------------------------------------------|
| `primary_hr_signal`         | Best heart-rate column for exercise detection                      |
| `primary_activity_signal`   | Best step/activity column (time-varying, not a counter)           |
| `activity_support_only`     | Activity proxy -- supports but does not anchor detection          |
| `primary_sleep_awake_signal`| Sleep/awake state, usable for filtering non-sleep windows         |
| `rr_support_signal`         | Respiratory rate -- physiological support                          |
| `spo2_support_signal`       | Oxygen saturation -- physiological support                         |
| `stress_support_signal`     | Stress/EDA -- physiological support, NOT an exercise signal alone |
| `clinical_stratifier`       | Static clinical feature -- for subgroup analysis only             |
| `glucose_response_only`     | Glucose -- reserved for downstream response analysis              |
| `id_column`                 | Participant or segment identifier                                  |
| `time_column`               | Timestamp or datetime column                                       |
| `not_usable`                | Too sparse, constant, or out of physiological range               |
| `possible_artifact`         | Suspected row counter or non-physiological artifact               |


In [11]:
def decide_role(col, metrics, modality, phys_ok):
    """Return (role, reason, usable_exercise, usable_response)."""
    if metrics is None:
        return "not_usable", "column not in sample", False, False

    miss_pct    = metrics.get("missing_pct", 100)
    time_vary   = metrics.get("time_varying_within_participant", False)
    const       = metrics.get("constant_per_participant",        False)
    poss_ctr    = metrics.get("possible_sequential_counter",     False)
    poss_cumul  = metrics.get("possible_cumulative_counter",     False)
    n_part      = metrics.get("n_participants_usable", 0) or 0
    n_part      = 0 if np.isnan(n_part) else int(n_part)

    enough_coverage = (miss_pct <= MISSING_THRESHOLD)

    # Metadata/auxiliary suffixes are never primary physiological signals
    _meta_suffixes = ("_count", "_n_records", "_device_availability", "_value_range",
                      "_baseline_date", "_days_to_cgm_start", "_n_records")
    is_meta = any(col.endswith(sfx) for sfx in _meta_suffixes)

    if modality == "participant":
        return "id_column", "participant identifier", False, False
    if modality == "time":
        return "time_column", "time/datetime identifier", False, False
    if modality == "clinical":
        return "clinical_stratifier", "static clinical feature for stratification", False, False

    if modality == "glucose":
        if enough_coverage and (phys_ok is True or phys_ok is np.nan):
            return "glucose_response_only", "glucose reserved for response analysis -- not exercise", False, True
        return "not_usable", f"glucose missing {miss_pct:.1f}% or implausible range", False, False

    # Heart rate
    if modality == "heart_rate":
        if is_meta:
            return "clinical_stratifier", "auxiliary count/metadata column -- not a physiological signal", False, False
        if not enough_coverage:
            return "not_usable", f"missing {miss_pct:.1f}% -- too sparse", False, False
        if const:
            return "not_usable", "constant per participant -- not physiologically varying", False, False
        if not time_vary:
            return "not_usable", "not time-varying within participants", False, False
        if phys_ok is False:
            return "not_usable", "values outside plausible HR range (30-220 bpm)", False, False
        if n_part < MIN_PARTICIPANTS_USABLE:
            return "not_usable", f"only {n_part} participants with >= 10 valid rows", False, False
        return "primary_hr_signal", "time-varying, plausible range, sufficient participant coverage", True, True

    # Steps / activity
    if modality == "steps":
        if not enough_coverage:
            return "not_usable", f"missing {miss_pct:.1f}%", False, False
        if poss_ctr:
            return "possible_artifact", "increments by 1 per row -- likely a row index, not wearable", False, False
        if poss_cumul and not time_vary:
            return "activity_support_only", "cumulative counter -- needs differencing before use", False, True
        if time_vary:
            return "primary_activity_signal", "time-varying interval activity signal", True, True
        return "activity_support_only", "low variance or ambiguous -- support use only", False, True

    # Calories
    if modality == "calories":
        if not enough_coverage:
            return "not_usable", f"missing {miss_pct:.1f}%", False, False
        if poss_ctr:
            return "possible_artifact", "sequential counter pattern", False, False
        if time_vary:
            return "activity_support_only", "per-interval calorie signal -- activity intensity support", False, True
        return "activity_support_only", "calorie/energy support -- possible cumulative", False, True

    # Respiratory rate
    if modality == "resp_rate":
        if is_meta:
            return "not_usable", "auxiliary count/metadata column", False, False
        if not enough_coverage or not time_vary:
            return "not_usable", f"missing {miss_pct:.1f}% or not time-varying", False, False
        if phys_ok is False:
            return "not_usable", "values outside plausible RR range (5-50 breaths/min)", False, False
        return "rr_support_signal", "respiratory rate -- physiological support signal", False, True

    # Sleep / awake
    if modality == "sleep":
        if not enough_coverage:
            return "not_usable", f"missing {miss_pct:.1f}%", False, False
        return "primary_sleep_awake_signal", "sleep/awake state -- usable for non-sleep filtering", False, True

    # Stress
    if modality == "stress":
        if is_meta:
            return "not_usable", "auxiliary count/metadata column", False, False
        if not enough_coverage:
            return "not_usable", f"missing {miss_pct:.1f}%", False, False
        return "stress_support_signal", "stress/EDA -- support signal, not standalone exercise detector", False, True

    # SpO2
    if modality == "spo2":
        if is_meta:
            return "not_usable", "auxiliary count/metadata column", False, False
        if not enough_coverage:
            return "not_usable", f"missing {miss_pct:.1f}%", False, False
        if phys_ok is None:
            return "spo2_support_signal", "SpO2 appears to be in fraction scale (0-1) -- rescale before use", False, True
        return "spo2_support_signal", "SpO2 -- physiological support signal", False, True

    return "not_usable", "unclassified modality", False, False


# Build the decision table
decision_rows = []
for col in ALL_CANDIDATE_COLS:
    metrics  = audit_results.get(col)
    mods     = classify_column(col)
    mod      = mods[0] if mods else "unknown"
    phys_ok  = phys_range_pass.get(col, np.nan)
    role, reason, use_ex, use_resp = decide_role(col, metrics, mod, phys_ok)

    row = {
        "column_name":                     col,
        "modality":                        mod,
        "dtype":                           (metrics.get("dtype", "n/a") if metrics else "n/a"),
        "missing_pct":                     (metrics.get("missing_pct", np.nan) if metrics else np.nan),
        "zero_frac":                       (metrics.get("zero_frac",   np.nan) if metrics else np.nan),
        "n_unique":                        (metrics.get("n_unique",    np.nan) if metrics else np.nan),
        "n_participants_usable":           (metrics.get("n_participants_usable", np.nan) if metrics else np.nan),
        "time_varying_within_participant": (metrics.get("time_varying_within_participant", np.nan) if metrics else np.nan),
        "constant_per_participant":        (metrics.get("constant_per_participant", np.nan) if metrics else np.nan),
        "possible_counter":                (metrics.get("possible_sequential_counter", np.nan) if metrics else np.nan),
        "plausible_range_pass":            phys_ok,
        "usable_for_exercise_detection":   use_ex,
        "usable_for_response_analysis":    use_resp,
        "recommended_role":                role,
        "reason":                          reason,
    }
    decision_rows.append(row)

decision_df = pd.DataFrame(decision_rows)
pd.set_option("display.max_colwidth", 60)
print(decision_df[["column_name", "modality", "missing_pct", "recommended_role",
                    "usable_for_exercise_detection", "usable_for_response_analysis"]].to_string(index=False))


                                 column_name    modality  missing_pct           recommended_role  usable_for_exercise_detection  usable_for_response_analysis
                            cgm_glucose_mean     glucose         1.24      glucose_response_only                          False                          True
                                   cgm_count     glucose         0.00                 not_usable                          False                         False
                       bmi_days_to_cgm_start     glucose       100.00                 not_usable                          False                         False
            c_peptide_ngml_days_to_cgm_start     glucose       100.00                 not_usable                          False                         False
clinical_diastolic_bp_mmhg_days_to_cgm_start     glucose       100.00                 not_usable                          False                         False
   clinical_resting_hr_bpm_days_to_cgm_start     glu

## 9. Participant-level signal availability

For each participant, compute how many rows they have and what fraction of those rows
contain valid (non-null) values for each key modality.
This reveals whether coverage issues are data-wide or limited to specific participants.


In [12]:
def compute_participant_availability(df):
    """Return a per-participant availability DataFrame."""
    if not id_cols or id_cols[0] not in df.columns:
        print("WARNING: No participant ID column found.  Skipping participant-level audit.")
        return pd.DataFrame()

    pid = id_cols[0]
    modality_best_col = {}
    for sig, cols_list in [
        ("hr",       hr_cols),
        ("activity", step_cols),
        ("rr",       rr_cols),
        ("sleep",    sleep_cols),
        ("spo2",     spo2_cols),
        ("stress",   stress_cols),
        ("glucose",  gluc_cols),
    ]:
        avail = [c for c in cols_list if c in df.columns]
        modality_best_col[sig] = avail[0] if avail else None

    records = []
    for pid_val, grp in df.groupby(pid):
        rec = {pid: pid_val, "n_rows": len(grp)}
        for sig, best_col in modality_best_col.items():
            if best_col:
                n_valid = int(grp[best_col].notna().sum())
                rec[f"has_{sig}"]         = n_valid > 0
                rec[f"frac_valid_{sig}"]  = round(n_valid / len(grp), 4) if len(grp) > 0 else 0.0
            else:
                rec[f"has_{sig}"]        = False
                rec[f"frac_valid_{sig}"] = 0.0
        records.append(rec)

    return pd.DataFrame(records)


part_avail_df = compute_participant_availability(df)
if len(part_avail_df) > 0:
    print(f"Participant-level availability table: {part_avail_df.shape}")
    print(part_avail_df.describe().round(3))
else:
    print("Participant-level availability table is empty.")


Participant-level availability table: (208, 16)
         n_rows  frac_valid_hr  frac_valid_activity  frac_valid_rr  \
count   208.000        208.000              208.000        208.000   
mean   2403.846          0.937                0.996          0.894   
std     619.918          0.083                0.049          0.101   
min      85.000          0.297                0.299          0.000   
25%    2387.750          0.929                1.000          0.873   
50%    2701.000          0.961                1.000          0.919   
75%    2738.250          0.976                1.000          0.947   
max    3327.000          0.993                1.000          1.000   

       frac_valid_sleep  frac_valid_spo2  frac_valid_stress  \
count           208.000          208.000            208.000   
mean              0.342            0.155              0.855   
std               0.087            0.056              0.105   
min               0.000            0.000              0.000   
25%   

## 10. Visual diagnostics

All figures are saved to `OUTPUT_DIR/figures/`.
Figures show:

1. Missingness by modality
2. Zero fraction by candidate wearable column
3. Heart rate distribution (primary HR column)
4. Activity/steps distribution (best activity column)
5. Respiratory rate distribution (if available)
6. SpO2 distribution (if available)
7. Participant availability summary (fraction of participants with each modality)
8. Example 24-hour timeline for one participant (HR, activity, sleep/awake, glucose)

The example timeline is included **only** to visually inspect data alignment and wearable availability.
Glucose is shown for alignment reference only -- it is NOT used for exercise analysis here.


In [13]:
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

MODALITY_DISPLAY = {
    "heart_rate": "Heart rate",
    "resp_rate":  "Respiratory rate",
    "steps":      "Steps / activity",
    "calories":   "Calories",
    "sleep":      "Sleep / awake",
    "stress":     "Stress",
    "spo2":       "SpO2",
    "glucose":    "Glucose (response only)",
    "time":       "Time",
    "participant":"Participant ID",
    "clinical":   "Clinical / static",
    "unknown":    "Unknown",
}

# ------------------------------------------------------------------ #
# 1. Missingness by modality
# ------------------------------------------------------------------ #
miss_by_mod = defaultdict(list)
for col, metrics in audit_results.items():
    mod = classify_column(col)[0]
    if mod in ("participant", "time", "clinical", "unknown"):
        continue
    if metrics:
        miss_by_mod[mod].append(metrics.get("missing_pct", 100))

mod_labels = sorted(miss_by_mod)
mod_means  = [np.mean(miss_by_mod[m]) for m in mod_labels]
mod_display= [MODALITY_DISPLAY.get(m, m) for m in mod_labels]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(mod_display, mod_means, color="steelblue", edgecolor="white")
ax.axvline(MISSING_THRESHOLD, color="crimson", linestyle="--", linewidth=1.2,
           label=f"Threshold {MISSING_THRESHOLD}%")
ax.set_xlabel("Mean missingness (%) across columns in modality")
ax.set_title("Missingness by wearable modality")
ax.legend()
for bar, val in zip(bars, mod_means):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=8)
plt.tight_layout()
fig.savefig(FIG_DIR / "01_missingness_by_modality.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "01_missingness_by_modality.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("Saved: 01_missingness_by_modality")

# ------------------------------------------------------------------ #
# 2. Zero fraction by candidate wearable column
# ------------------------------------------------------------------ #
zero_cols   = []
zero_vals   = []
for col in ALL_CANDIDATE_COLS:
    mod = classify_column(col)[0]
    if mod in ("participant", "time", "clinical", "glucose", "unknown"):
        continue
    m = audit_results.get(col)
    if m and not np.isnan(m.get("zero_frac", np.nan)):
        zero_cols.append(col)
        zero_vals.append(m["zero_frac"])

if zero_cols:
    sort_idx   = np.argsort(zero_vals)[::-1]
    zero_cols  = [zero_cols[i] for i in sort_idx]
    zero_vals  = [zero_vals[i] for i in sort_idx]

    fig, ax = plt.subplots(figsize=(10, max(4, len(zero_cols) * 0.35)))
    ax.barh(zero_cols, zero_vals, color="darkorange", edgecolor="white")
    ax.set_xlabel("Zero fraction (fraction of non-null values equal to zero)")
    ax.set_title("Zero inflation by wearable column")
    ax.set_xlim(0, 1)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "02_zero_fraction_by_column.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "02_zero_fraction_by_column.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print("Saved: 02_zero_fraction_by_column")

# ------------------------------------------------------------------ #
# 3. Heart rate distribution
# ------------------------------------------------------------------ #
best_hr = next((c for c in hr_cols if c in df.columns), None)
if best_hr:
    hr_vals = df[best_hr].dropna()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(hr_vals, bins=60, color="tomato", edgecolor="white", linewidth=0.4)
    ax.axvline(30,  color="navy", linestyle="--", linewidth=1, label="Plausible lower (30 bpm)")
    ax.axvline(220, color="navy", linestyle="--", linewidth=1, label="Plausible upper (220 bpm)")
    ax.set_xlabel(f"{best_hr}")
    ax.set_ylabel("Count (sample)")
    ax.set_title("Heart rate distribution")
    ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "03_hr_distribution.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "03_hr_distribution.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: 03_hr_distribution  (column={best_hr})")
else:
    print("No heart rate column found -- skipping HR distribution plot")

# ------------------------------------------------------------------ #
# 4. Activity / steps distribution
# ------------------------------------------------------------------ #
best_act = next((c for c in step_cols if c in df.columns), None)
if best_act:
    act_vals = df[best_act].dropna()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(act_vals, bins=60, color="seagreen", edgecolor="white", linewidth=0.4)
    ax.set_xlabel(f"{best_act}")
    ax.set_ylabel("Count (sample)")
    ax.set_title("Activity / steps distribution")
    plt.tight_layout()
    fig.savefig(FIG_DIR / "04_activity_distribution.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "04_activity_distribution.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: 04_activity_distribution  (column={best_act})")
else:
    print("No activity/steps column found -- skipping activity distribution plot")

# ------------------------------------------------------------------ #
# 5. Respiratory rate distribution
# ------------------------------------------------------------------ #
best_rr = next((c for c in rr_cols if c in df.columns), None)
if best_rr:
    rr_vals = df[best_rr].dropna()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(rr_vals, bins=60, color="mediumpurple", edgecolor="white", linewidth=0.4)
    ax.axvline(5,  color="navy", linestyle="--", linewidth=1, label="Plausible lower (5)")
    ax.axvline(50, color="navy", linestyle="--", linewidth=1, label="Plausible upper (50)")
    ax.set_xlabel(f"{best_rr}")
    ax.set_ylabel("Count (sample)")
    ax.set_title("Respiratory rate distribution")
    ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "05_rr_distribution.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "05_rr_distribution.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: 05_rr_distribution  (column={best_rr})")
else:
    print("No respiratory rate column found -- skipping RR distribution plot")

# ------------------------------------------------------------------ #
# 6. SpO2 distribution
# ------------------------------------------------------------------ #
best_spo2 = next((c for c in spo2_cols if c in df.columns), None)
if best_spo2:
    sp_vals = df[best_spo2].dropna()
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(sp_vals, bins=60, color="cornflowerblue", edgecolor="white", linewidth=0.4)
    ax.axvline(50,  color="navy", linestyle="--", linewidth=1, label="Plausible lower (50%)")
    ax.axvline(100, color="navy", linestyle="--", linewidth=1, label="Plausible upper (100%)")
    ax.set_xlabel(f"{best_spo2}")
    ax.set_ylabel("Count (sample)")
    ax.set_title("SpO2 distribution")
    ax.legend(fontsize=8)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "06_spo2_distribution.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "06_spo2_distribution.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: 06_spo2_distribution  (column={best_spo2})")
else:
    print("No SpO2 column found -- skipping SpO2 distribution plot")

# ------------------------------------------------------------------ #
# 7. Participant availability summary
# ------------------------------------------------------------------ #
if len(part_avail_df) > 0:
    sig_cols  = [c for c in part_avail_df.columns if c.startswith("has_")]
    sig_labels= [c.replace("has_", "") for c in sig_cols]
    frac_have = [part_avail_df[c].mean() for c in sig_cols]

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(sig_labels, frac_have, color="slategray", edgecolor="white")
    ax.set_xlabel("Fraction of participants with at least one valid value")
    ax.set_title("Participant coverage by modality")
    ax.set_xlim(0, 1)
    for bar, val in zip(bars, frac_have):
        ax.text(min(bar.get_width() + 0.01, 0.95),
                bar.get_y() + bar.get_height() / 2,
                f"{val:.0%}", va="center", fontsize=8)
    plt.tight_layout()
    fig.savefig(FIG_DIR / "07_participant_availability.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "07_participant_availability.png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    print("Saved: 07_participant_availability")
else:
    print("Participant availability table empty -- skipping coverage plot")

print("\nAll diagnostic figures saved to:", FIG_DIR)


Saved: 01_missingness_by_modality
Saved: 02_zero_fraction_by_column
Saved: 03_hr_distribution  (column=heart_rate_mean)
Saved: 04_activity_distribution  (column=activity_stage_walking)
Saved: 05_rr_distribution  (column=respiratory_rate_mean)
Saved: 06_spo2_distribution  (column=oxygen_saturation_mean)
Saved: 07_participant_availability

All diagnostic figures saved to: outputs/exercise_wearable_audit/figures


## 11. Example 24-hour participant timeline

This plot shows the wearable signals for a single participant across one day (or the first 24
hours of available data if day boundaries cannot be inferred).
The glucose trace is included **only** to visually confirm that the wearable timestamps
align with the CGM stream.  Glucose values are NOT used for exercise analysis.


In [14]:
# Sleep stage columns and their display properties
SLEEP_STAGE_COLS = [
    "sleep_stage_deep",
    "sleep_stage_light",
    "sleep_stage_rem",
    "sleep_stage_awake",
    "sleep_stage_unknown",
]
SLEEP_STAGE_COLORS = {
    "sleep_stage_deep":    ("#1a3a6b", "Deep sleep"),
    "sleep_stage_light":   ("#6baed6", "Light sleep"),
    "sleep_stage_rem":     ("#9e6bbf", "REM"),
    "sleep_stage_awake":   ("#f4a44a", "Awake within session"),
    "sleep_stage_unknown": ("#bbbbbb", "Unknown stage"),
}

# Preferred activity columns for the timeline (steps/min is clearest)
_ACT_PREF = ["activity_steps_per_min", "activity_intensity_score",
             "activity_stage_walking", "activity_stage_running"]


def add_sleep_session_flag(sub):
    """Add is_in_sleep_session and is_asleep to a participant slice.

    is_in_sleep_session = True when the wearable is inside a tracked sleep session
    (all sleep stage fractions sum > 0).  When False the tracker is idle (daytime);
    sleep_stage_awake == 0 in that case does NOT mean the person is asleep.

    is_asleep = inside a sleep session AND less than 50% of the minute is awake.
    """
    avail = [c for c in SLEEP_STAGE_COLS if c in sub.columns]
    if not avail:
        sub = sub.copy()
        sub["is_in_sleep_session"] = False
        sub["is_asleep"]           = False
        return sub, avail
    sub = sub.copy()
    sub["is_in_sleep_session"] = sub[avail].sum(axis=1) > 0.0
    awake_col = "sleep_stage_awake" if "sleep_stage_awake" in sub.columns else None
    if awake_col:
        sub["is_asleep"] = sub["is_in_sleep_session"] & (sub[awake_col] < 0.5)
    else:
        sub["is_asleep"] = sub["is_in_sleep_session"].copy()
    return sub, avail


def pick_participant_with_sleep(df, pid_col):
    """Return the participant with the most sleep-session minutes in the sample."""
    check_col = next((c for c in SLEEP_STAGE_COLS if c in df.columns), None)
    if check_col:
        totals = df.groupby(pid_col)[check_col].sum()
        cands  = totals[totals > 0]
        if len(cands) > 0:
            return cands.idxmax()
    return df.groupby(pid_col).size().idxmax()


def plot_participant_timeline(df, pid_val=1800, n_hours=24):
    """Plot HR, activity, real sleep stages (stacked), and glucose for one participant.

    Sleep panel reads as follows:
    - Yellow background  : daytime -- tracker idle, all stage fractions are 0,
                           sleep_stage_awake=0 here does NOT mean the person is asleep.
    - Dark blue fill     : deep sleep (inside a tracked session).
    - Cornflower fill    : light sleep.
    - Purple fill        : REM sleep.
    - Orange fill        : awake within a sleep session (nocturnal awakening).
    - Gray fill          : unknown stage inside a session.
    Fractions in each minute sum to 1.0 when the tracker is active.
    """
    if not id_cols or id_cols[0] not in df.columns:
        print("No participant ID column -- cannot draw timeline.")
        return
    if not time_cols or time_cols[0] not in df.columns:
        print("No time column found -- cannot draw timeline.")
        return

    pid_col = id_cols[0]
    tcol    = time_cols[0]

    if pid_val is None:
        pid_val = pick_participant_with_sleep(df, pid_col)

    sub = df[df[pid_col] == pid_val].copy()
    try:
        sub[tcol] = pd.to_datetime(sub[tcol], utc=True)
        sub = sub.sort_values(tcol).reset_index(drop=True)
        t0  = sub[tcol].iloc[0]
        sub = sub[sub[tcol] <= t0 + pd.Timedelta(hours=n_hours)].copy()
        x   = sub[tcol]
    except Exception:
        sub = sub.sort_values(tcol).head(n_hours * 60).copy()
        x   = np.arange(len(sub))

    sub, avail_sleep = add_sleep_session_flag(sub)

    best_hr   = next((c for c in hr_cols   if c in sub.columns), None)
    best_act  = next((c for c in _ACT_PREF if c in sub.columns), None)
    if best_act is None:
        best_act = next((c for c in step_cols if c in sub.columns), None)
    best_gluc = next((c for c in gluc_cols  if c in sub.columns), None)
    has_sleep = len(avail_sleep) > 0

    panels = []
    if best_hr:   panels.append("hr")
    if best_act:  panels.append("act")
    if has_sleep: panels.append("sleep")
    if best_gluc: panels.append("gluc")

    if not panels:
        print("No plottable signals for this participant.")
        return

    fig, axes = plt.subplots(len(panels), 1, figsize=(13, 2.8 * len(panels)), sharex=True)
    if len(panels) == 1:
        axes = [axes]

    ai = 0

    # -- Heart rate --
    if "hr" in panels:
        ax = axes[ai]; ai += 1
        ax.plot(x, sub[best_hr].values, color="tomato", linewidth=0.7, label=best_hr)
        ax.set_ylabel("HR (bpm)", fontsize=9)
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(axis="y", alpha=0.3)

    # -- Activity --
    if "act" in panels:
        ax = axes[ai]; ai += 1
        ax.plot(x, sub[best_act].values, color="seagreen", linewidth=0.7, label=best_act)
        ax.set_ylabel("Activity", fontsize=9)
        ax.legend(fontsize=7, loc="upper right")
        ax.grid(axis="y", alpha=0.3)

    # -- Sleep stages (stacked fractions) --
    if "sleep" in panels:
        ax = axes[ai]; ai += 1

        # Yellow background where tracker is idle (daytime)
        daytime = ~sub["is_in_sleep_session"].values
        ax.fill_between(x, 0, 1, where=daytime,
                        color="#fffbe6", alpha=0.9,
                        label="Daytime (tracker idle)", linewidth=0)

        # Stacked sleep stage fractions
        bottom = np.zeros(len(sub))
        for col in SLEEP_STAGE_COLS:
            if col not in sub.columns:
                continue
            vals          = sub[col].fillna(0).values
            color, label  = SLEEP_STAGE_COLORS[col]
            ax.fill_between(x, bottom, bottom + vals,
                            color=color, alpha=0.88, label=label, linewidth=0)
            bottom = bottom + vals

        ax.set_ylim(0, 1.05)
        ax.set_ylabel("Sleep stage\n(fraction)", fontsize=9)
        ax.set_yticks([0, 0.5, 1.0])
        ax.yaxis.set_tick_params(labelsize=8)
        ax.axhline(0.5, color="gray", linewidth=0.4, linestyle=":")

        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles, labels, fontsize=6, loc="upper right", ncol=3, framealpha=0.9)
        ax.grid(axis="y", alpha=0.2)

        note = (
            "Yellow = daytime (tracker idle):  sleep_stage_awake=0 here does NOT mean asleep.\n"
            "Colored stacks = fractions inside a tracked sleep session (sum to 1.0 per minute)."
        )
        ax.text(0.0, -0.32, note, transform=ax.transAxes, fontsize=6.5,
                color="dimgray", verticalalignment="top",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))

    # -- Glucose (alignment reference only) --
    if "gluc" in panels:
        ax = axes[ai]; ai += 1
        ax.plot(x, sub[best_gluc].values, color="steelblue", linewidth=0.7, label=best_gluc)
        ax.set_ylabel("Glucose\n(mg/dL)", fontsize=9)
        ax.legend(fontsize=7, loc="upper right")
        ax.set_title("Glucose shown for timestamp alignment only -- not for exercise analysis",
                     fontsize=7, style="italic", color="gray", pad=2)
        ax.grid(axis="y", alpha=0.3)

    axes[-1].set_xlabel("Time (UTC)", fontsize=9)
    fig.suptitle(
        f"Example {n_hours}-hour timeline  --  participant {pid_val}\n"
        f"Sleep fractions stack to 1.0 inside a session; yellow background = tracker idle (daytime)",
        fontsize=9, y=1.01,
    )
    plt.tight_layout()
    fig.savefig(FIG_DIR / "08_example_24h_timeline.pdf", bbox_inches="tight")
    fig.savefig(FIG_DIR / "08_example_24h_timeline.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: 08_example_24h_timeline  (participant={pid_val})")
    print(f"  Rows in window             : {len(sub)}")
    print(f"  Sleep-session minutes      : {int(sub['is_in_sleep_session'].sum())}")
    print(f"  Asleep minutes             : {int(sub['is_asleep'].sum())}")
    print(f"  Awake-in-session minutes   : {int((sub['is_in_sleep_session'] & ~sub['is_asleep']).sum())}")


plot_participant_timeline(df)


Saved: 08_example_24h_timeline  (participant=1800)
  Rows in window             : 0
  Sleep-session minutes      : 0
  Asleep minutes             : 0
  Awake-in-session minutes   : 0


In [15]:
# ---------------------------------------------------------------
# Plot timeline for a specific participant (participant 1800)
# 1800 is not in the 500k-row sample, so we load their rows
# directly from the full parquet file before plotting.
# Change TARGET_PID to plot any other participant.
# ---------------------------------------------------------------

TARGET_PID = "1800"

_COLS_PID = [
    "participant_id", "timestamp_local", "timezone",
    "heart_rate_mean", "heart_rate_std", "heart_rate_min", "heart_rate_max",
    "activity_steps_per_min", "activity_intensity_score",
    "activity_stage_walking", "activity_stage_running",
    "sleep_stage_deep", "sleep_stage_light", "sleep_stage_rem",
    "sleep_stage_awake", "sleep_stage_unknown",
    "cgm_glucose_mean", "respiratory_rate_mean",
]
_existing  = set(pq.ParquetFile(DATA_PATH).schema_arrow.names)
_COLS_PID  = [c for c in _COLS_PID if c in _existing]

print(f"Loading rows for participant {TARGET_PID} from full file ...")
_batches = []
for _b in pq.ParquetFile(DATA_PATH).iter_batches(batch_size=100_000, columns=_COLS_PID):
    _df_b = _b.to_pandas()
    _sub  = _df_b[_df_b["participant_id"] == TARGET_PID]
    if len(_sub):
        _batches.append(_sub)

if not _batches:
    print(f"Participant {TARGET_PID} not found in {DATA_PATH}.")
else:
    df_pid = pd.concat(_batches).reset_index(drop=True)
    print(f"Loaded {len(df_pid):,} rows  |  "
          f"{df_pid['timestamp_local'].min()}  to  {df_pid['timestamp_local'].max()}")

    df_pid, _ = add_sleep_session_flag(df_pid)
    print(f"Sleep-session minutes : {int(df_pid['is_in_sleep_session'].sum())} / {len(df_pid)}")

    # Plot and save to a participant-specific filename
    plot_participant_timeline(df_pid, pid_val=TARGET_PID, n_hours=24)

    import shutil as _sh
    for _ext in (".pdf", ".png"):
        _src = FIG_DIR / f"08_example_24h_timeline{_ext}"
        _dst = FIG_DIR / f"08_example_24h_timeline_pid{TARGET_PID}{_ext}"
        if _src.exists():
            _sh.copy2(_src, _dst)
    print(f"Saved: 08_example_24h_timeline_pid{TARGET_PID}.pdf / .png")


Loading rows for participant 1800 from full file ...
Loaded 2,713 rows  |  2025-05-01 00:00:00-07:00  to  2025-05-10 10:00:00-07:00
Sleep-session minutes : 1060 / 2713
Saved: 08_example_24h_timeline  (participant=1800)
  Rows in window             : 1440
  Sleep-session minutes      : 547
  Asleep minutes             : 426
  Awake-in-session minutes   : 121
Saved: 08_example_24h_timeline_pid1800.pdf / .png


## 12. Final recommendation

Based on all metrics above, the notebook prints a clear recommendation about
which analysis path is supported.

**Rules:**

- Exercise-like episode detection is feasible if: usable HR + usable activity + awake state.
- Physiological-arousal response modeling is feasible if: usable HR + sleep/awake, even without true activity.
- Stop / request raw wearable files if: HR is missing or unusable, or no time-varying wearable exists.


In [16]:
# Identify the best column for each role from the decision table
def best_col_with_role(role_prefix):
    """Return the first column whose recommended role starts with role_prefix."""
    for row in decision_rows:
        if row["recommended_role"].startswith(role_prefix):
            return row["column_name"]
    return None

chosen_hr       = best_col_with_role("primary_hr_signal")
chosen_activity = best_col_with_role("primary_activity_signal")
chosen_rr       = best_col_with_role("rr_support_signal")
chosen_sleep    = best_col_with_role("primary_sleep_awake_signal")
chosen_spo2     = best_col_with_role("spo2_support_signal")
chosen_glucose  = best_col_with_role("glucose_response_only")
chosen_id   = id_cols[0] if id_cols else None

# Prefer timestamp-style time columns over timezone or date-string columns
_time_pref  = [c for c in time_cols if "timestamp" in c.lower()]
_time_pref += [c for c in time_cols if "datetime" in c.lower() and c not in _time_pref]
_time_pref += [c for c in time_cols if c not in _time_pref]
chosen_time = _time_pref[0] if _time_pref else None

hr_ok       = chosen_hr       is not None
activity_ok = chosen_activity is not None
sleep_ok    = chosen_sleep    is not None

if hr_ok and activity_ok:
    if sleep_ok:
        path = "EXERCISE-LIKE EPISODE DETECTION"
        detail = (
            "Usable heart rate, activity signal, and sleep/awake state are all present.\n"
            "A wearable-defined exercise episode detector can be built."
        )
    else:
        path = "EXERCISE-LIKE EPISODE DETECTION (no sleep filter)"
        detail = (
            "Usable heart rate and activity signal are present, but no sleep/awake column found.\n"
            "Episodes will need to be filtered post-hoc or with time-of-day heuristics."
        )
elif hr_ok and not activity_ok:
    path = "PHYSIOLOGICAL-AROUSAL RESPONSE MODELING"
    detail = (
        "Usable heart rate found, but no true per-interval activity column detected.\n"
        "Arousal response modeling using HR (and optionally RR, SpO2, stress) is the recommended path."
    )
else:
    path = "STOP -- REQUEST RAW WEARABLE FILES"
    detail = (
        "Heart rate is missing or unusable.  No time-varying wearable signal is available.\n"
        "Request raw device-level files before proceeding."
    )

SEP = "=" * 65
print(SEP)
print(f"  RECOMMENDED ANALYSIS PATH: {path}")
print(SEP)
print(detail)
print()
print(f"  Primary HR column          : {chosen_hr}")
print(f"  Primary activity column    : {chosen_activity}")
print(f"  Respiratory rate column    : {chosen_rr}")
print(f"  Sleep/awake column         : {chosen_sleep}")
print(f"  SpO2 column                : {chosen_spo2}")
print(f"  Glucose column (later use) : {chosen_glucose}")
print(f"  Participant ID column      : {chosen_id}")
print(f"  Time column                : {chosen_time}")
print(SEP)


  RECOMMENDED ANALYSIS PATH: EXERCISE-LIKE EPISODE DETECTION
Usable heart rate, activity signal, and sleep/awake state are all present.
A wearable-defined exercise episode detector can be built.

  Primary HR column          : heart_rate_mean
  Primary activity column    : activity_stage_walking
  Respiratory rate column    : respiratory_rate_mean
  Sleep/awake column         : sleep_stage_awake
  SpO2 column                : None
  Glucose column (later use) : cgm_glucose_mean
  Participant ID column      : participant_id
  Time column                : timestamp_local


## 13. Save artifacts

All audit outputs are saved to `OUTPUT_DIR` so they can be inspected offline or
loaded by a downstream exercise detector notebook.


In [17]:
# wearable_column_decision_table.csv
decision_df.to_csv(OUTPUT_DIR / "wearable_column_decision_table.csv", index=False)
print(f"Saved: {OUTPUT_DIR}/wearable_column_decision_table.csv  ({len(decision_df)} rows)")

# wearable_availability_audit.csv -- full per-column numeric audit
audit_df = pd.DataFrame([
    {"column_name": col, **{k: v for k, v in (metrics or {}).items()}}
    for col, metrics in audit_results.items()
])
audit_df.to_csv(OUTPUT_DIR / "wearable_availability_audit.csv", index=False)
print(f"Saved: {OUTPUT_DIR}/wearable_availability_audit.csv  ({len(audit_df)} rows)")

# participant_wearable_availability.csv
if len(part_avail_df) > 0:
    part_avail_df.to_csv(OUTPUT_DIR / "participant_wearable_availability.csv", index=False)
    print(f"Saved: {OUTPUT_DIR}/participant_wearable_availability.csv  ({len(part_avail_df)} rows)")

# wearable_audit_summary.json
def _safe(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (np.bool_,)):
        return bool(v)
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return float(v)
    return v

summary = {
    "data_path":              str(DATA_PATH),
    "static_path":            str(STATIC_PATH),
    "sample_n":               len(df),
    "is_sample":              IS_SAMPLE,
    "total_rows_in_file":     int(dyn_nrows),
    "recommended_path":       path,
    "chosen_hr_column":       chosen_hr,
    "chosen_activity_column": chosen_activity,
    "chosen_rr_column":       chosen_rr,
    "chosen_sleep_column":    chosen_sleep,
    "chosen_spo2_column":     chosen_spo2,
    "chosen_glucose_column":  chosen_glucose,
    "chosen_id_column":       chosen_id,
    "chosen_time_column":     chosen_time,
    "decision_table": [
        {k: _safe(v) for k, v in row.items()} for row in decision_rows
    ],
}
with open(OUTPUT_DIR / "wearable_audit_summary.json", "w") as fh:
    json.dump(summary, fh, indent=2)
print(f"Saved: {OUTPUT_DIR}/wearable_audit_summary.json")
print("\nAll artifacts saved.")


Saved: outputs/exercise_wearable_audit/wearable_column_decision_table.csv  (77 rows)
Saved: outputs/exercise_wearable_audit/wearable_availability_audit.csv  (77 rows)
Saved: outputs/exercise_wearable_audit/participant_wearable_availability.csv  (208 rows)
Saved: outputs/exercise_wearable_audit/wearable_audit_summary.json

All artifacts saved.


## 14. Batch timelines for all cohort participants

Generate one 24-hour timeline per participant for the **1591 selected cohort participants**.
All plots are saved as individual PNGs and combined into a single multi-page PDF.

The cohort is loaded from `enriched_multimodal/cohort.csv`.
Full wearable data is loaded once from the parquet file, then split by participant.


In [18]:
import matplotlib.backends.backend_pdf as _mpdf
from pathlib import Path as _Path

# ---------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------
COHORT_CSV  = _Path("/home/myriamcharfeddine/CGM/Data/enriched_multimodal/cohort.csv")
BATCH_OUTDIR = OUTPUT_DIR / "timelines_all_participants"
BATCH_OUTDIR.mkdir(parents=True, exist_ok=True)
BATCH_N_HOURS = 24   # hours to show per participant; change to 48 or 72 for more

# ---------------------------------------------------------------
# Load cohort participant IDs
# ---------------------------------------------------------------
cohort_df  = pd.read_csv(COHORT_CSV)
cohort_pids = set(cohort_df["participant_id"].astype(str).tolist())
print(f"Cohort participants : {len(cohort_pids)}")
print(f"Strata              : {cohort_df['stratum'].value_counts().to_dict()}")

# ---------------------------------------------------------------
# Load all cohort rows from the full parquet (one pass)
# ---------------------------------------------------------------
BATCH_COLS = [
    "participant_id", "timestamp_local",
    "heart_rate_mean",
    "activity_steps_per_min", "activity_stage_walking", "activity_stage_running",
    "activity_intensity_score",
    "sleep_stage_deep", "sleep_stage_light", "sleep_stage_rem",
    "sleep_stage_awake", "sleep_stage_unknown",
    "cgm_glucose_mean",
]
_existing = set(pq.ParquetFile(DATA_PATH).schema_arrow.names)
BATCH_COLS = [c for c in BATCH_COLS if c in _existing]

print(f"\nLoading {len(BATCH_COLS)} columns for all {len(cohort_pids)} cohort participants ...")
_batches = []
for _b in pq.ParquetFile(DATA_PATH).iter_batches(batch_size=200_000, columns=BATCH_COLS):
    _df_b = _b.to_pandas()
    _sub  = _df_b[_df_b["participant_id"].astype(str).isin(cohort_pids)]
    if len(_sub):
        _batches.append(_sub)

df_all = pd.concat(_batches).reset_index(drop=True)
df_all["participant_id"] = df_all["participant_id"].astype(str)
print(f"Loaded {len(df_all):,} rows covering {df_all['participant_id'].nunique()} participants")

# ---------------------------------------------------------------
# Add sleep session flag to the full frame
# ---------------------------------------------------------------
df_all, _ = add_sleep_session_flag(df_all)

# ---------------------------------------------------------------
# Batch plot loop
# ---------------------------------------------------------------
SLEEP_COLS_BATCH = [c for c in SLEEP_STAGE_COLS if c in df_all.columns]
pdf_path  = BATCH_OUTDIR / "all_cohort_timelines.pdf"
png_dir   = BATCH_OUTDIR / "png"
png_dir.mkdir(exist_ok=True)

pid_list   = sorted(df_all["participant_id"].unique())
n_total    = len(pid_list)
n_done     = 0
n_skipped  = 0
n_no_sleep = 0

# Join stratum from cohort_df for title annotation
stratum_map = cohort_df.set_index(cohort_df["participant_id"].astype(str))["stratum"].to_dict()

try:
    from tqdm.auto import tqdm as _tqdm
    pid_iter = _tqdm(pid_list, desc="Generating timelines")
except ImportError:
    pid_iter = pid_list

with _mpdf.PdfPages(pdf_path) as pdf:
    for pid in pid_iter:
        sub = df_all[df_all["participant_id"] == pid].copy()
        if len(sub) < 10:
            n_skipped += 1
            continue

        # Sort and take first BATCH_N_HOURS of data
        try:
            sub["timestamp_local"] = pd.to_datetime(sub["timestamp_local"], utc=True)
            sub = sub.sort_values("timestamp_local").reset_index(drop=True)
            t0  = sub["timestamp_local"].iloc[0]
            sub = sub[sub["timestamp_local"] <= t0 + pd.Timedelta(hours=BATCH_N_HOURS)].copy()
            x   = sub["timestamp_local"]
        except Exception:
            sub = sub.sort_values("timestamp_local").head(BATCH_N_HOURS * 60).copy()
            x   = np.arange(len(sub))

        if len(sub) < 5:
            n_skipped += 1
            continue

        has_sleep_data = (sub[SLEEP_COLS_BATCH].sum(axis=1) > 0).any() if SLEEP_COLS_BATCH else False
        if not has_sleep_data:
            n_no_sleep += 1

        stratum = stratum_map.get(str(pid), "unknown")

        # Build panels
        best_hr  = next((c for c in ["heart_rate_mean"] if c in sub.columns), None)
        best_act = next((c for c in _ACT_PREF if c in sub.columns), None)
        has_slp  = len(SLEEP_COLS_BATCH) > 0
        best_gluc= next((c for c in ["cgm_glucose_mean"] if c in sub.columns), None)

        panels = []
        if best_hr:   panels.append("hr")
        if best_act:  panels.append("act")
        if has_slp:   panels.append("sleep")
        if best_gluc: panels.append("gluc")

        if not panels:
            n_skipped += 1
            continue

        fig, axes = plt.subplots(len(panels), 1, figsize=(12, 2.4 * len(panels)), sharex=True)
        if len(panels) == 1:
            axes = [axes]

        ai = 0

        # -- HR --
        if "hr" in panels:
            ax = axes[ai]; ai += 1
            ax.plot(x, sub[best_hr].values, color="tomato", linewidth=0.6)
            ax.set_ylabel("HR (bpm)", fontsize=8)
            ax.grid(axis="y", alpha=0.25)
            ax.tick_params(labelsize=7)

        # -- Activity --
        if "act" in panels:
            ax = axes[ai]; ai += 1
            ax.plot(x, sub[best_act].values, color="seagreen", linewidth=0.6)
            ax.set_ylabel("Steps/min", fontsize=8)
            ax.grid(axis="y", alpha=0.25)
            ax.tick_params(labelsize=7)

        # -- Sleep stages (stacked) --
        if "sleep" in panels:
            ax = axes[ai]; ai += 1
            daytime = ~sub["is_in_sleep_session"].values
            ax.fill_between(x, 0, 1, where=daytime,
                            color="#fffbe6", alpha=0.9, linewidth=0)
            bottom = np.zeros(len(sub))
            for col in SLEEP_STAGE_COLS:
                if col not in sub.columns:
                    continue
                vals         = sub[col].fillna(0).values
                color, label = SLEEP_STAGE_COLORS[col]
                ax.fill_between(x, bottom, bottom + vals,
                                color=color, alpha=0.85, linewidth=0)
                bottom = bottom + vals
            ax.set_ylim(0, 1.05)
            ax.set_ylabel("Sleep stage", fontsize=8)
            ax.set_yticks([0, 0.5, 1.0])
            ax.tick_params(labelsize=7)
            ax.grid(axis="y", alpha=0.2)

        # -- Glucose --
        if "gluc" in panels:
            ax = axes[ai]; ai += 1
            ax.plot(x, sub[best_gluc].values, color="steelblue", linewidth=0.6)
            ax.set_ylabel("Glucose\n(mg/dL)", fontsize=8)
            ax.grid(axis="y", alpha=0.25)
            ax.tick_params(labelsize=7)

        axes[-1].set_xlabel("Time (UTC)", fontsize=8)
        sleep_min  = int(sub["is_in_sleep_session"].sum())
        asleep_min = int(sub["is_asleep"].sum())
        fig.suptitle(
            f"Participant {pid}  |  stratum: {stratum}  |  "
            f"rows: {len(sub)}  |  sleep-session: {sleep_min} min  |  asleep: {asleep_min} min",
            fontsize=8, y=1.005,
        )
        plt.tight_layout()

        # Save PNG (individual)
        png_path = png_dir / f"timeline_pid{pid}.png"
        fig.savefig(png_path, dpi=100, bbox_inches="tight")
        # Add to multi-page PDF
        pdf.savefig(fig, bbox_inches="tight")
        plt.close(fig)
        n_done += 1

print(f"\nDone.")
print(f"  Participants plotted  : {n_done}")
print(f"  Skipped (< 5 rows)   : {n_skipped}")
print(f"  No sleep data        : {n_no_sleep}")
print(f"\nMulti-page PDF : {pdf_path}")
print(f"Individual PNGs: {png_dir}  ({n_done} files)")


Cohort participants : 1591
Strata              : {'normoglycemia': 564, 'T2D-oral': 458, 'prediabetes': 405, 'T2D-insulin': 164}

Loading 13 columns for all 1591 cohort participants ...
Loaded 4,037,026 rows covering 1591 participants


Generating timelines: 100%|██████████| 1591/1591 [29:40<00:00,  1.12s/it]


Done.
  Participants plotted  : 1591
  Skipped (< 5 rows)   : 0
  No sleep data        : 19

Multi-page PDF : outputs/exercise_wearable_audit/timelines_all_participants/all_cohort_timelines.pdf
Individual PNGs: outputs/exercise_wearable_audit/timelines_all_participants/png  (1591 files)


## Next step after this audit

Build a high-precision exercise-like episode detector only if the audit confirms
usable heart rate plus activity, or usable heart rate plus physiological support signals.
